In [1]:
"""
Phase 1 - EDA & stationarity.

Run this directly (`python notebooks/01_eda.py` from the repo root) or open it
in VS Code with the Jupyter extension and run cell-by-cell (the `# %%` markers
define cells). Either way it saves every plot to results/figures/ and prints
the numeric tests to the console, so you don't need an interactive window to
see the output.

What this phase answers, before touching any model:
  1. What does the series actually look like - trend? seasonality? shocks?
  2. Is it stationary? (ARIMA needs to know this to pick `d`)
  3. What do ACF/PACF suggest about AR/MA order?
"""
import os
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Works both as a script (__file__ defined) and inside a Jupyter kernel
# (__file__ undefined - falls back to the notebook's working directory,
# which Jupyter sets to the folder the .ipynb lives in, i.e. notebooks/).
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

FIG_DIR = os.path.join(BASE_DIR, "..", "results", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

In [2]:
df = pd.read_csv(
    os.path.join(BASE_DIR, "..", "data", "people_analytics_monthly.csv"),
    parse_dates=["Date"],
)
df = df.set_index("Date").asfreq("MS")
print(df.shape)
print(df.head())

(240, 9)
            Headcount  Hires  Voluntary_Terminations  \
Date                                                   
2006-01-01        121      3                       2   
2006-02-01        121      3                       3   
2006-03-01        121      3                       3   
2006-04-01        121      2                       1   
2006-05-01        124      5                       0   

            Involuntary_Terminations  Total_Terminations  Attrition_Rate_Pct  \
Date                                                                           
2006-01-01                         0                   2                1.66   
2006-02-01                         0                   3                2.48   
2006-03-01                         0                   3                2.48   
2006-04-01                         1                   2                1.65   
2006-05-01                         2                   2                1.63   

            Avg_Engagement_Score  Avg

In [3]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
df["Headcount"].plot(ax=axes[0], title="Monthly Headcount")
df["Attrition_Rate_Pct"].plot(ax=axes[1], title="Monthly Attrition Rate (%)", color="firebrick")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "01_raw_series.png"), dpi=150)
plt.close()
print("Saved 01_raw_series.png - look for: overall trend, visible dips (2009 / 2020), any yearly wobble.")

Saved 01_raw_series.png - look for: overall trend, visible dips (2009 / 2020), any yearly wobble.


In [4]:
for col in ["Headcount", "Attrition_Rate_Pct"]:
    decomp = seasonal_decompose(df[col], model="additive", period=12)
    fig = decomp.plot()
    fig.set_size_inches(10, 7)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"02_decomposition_{col}.png"), dpi=150)
    plt.close()
print("Saved decomposition plots - check the 'seasonal' panel: is there a repeating yearly pattern? "
      "check 'resid': is what's left over roughly noise, or does it still show the 2009/2020 dips?")

Saved decomposition plots - check the 'seasonal' panel: is there a repeating yearly pattern? check 'resid': is what's left over roughly noise, or does it still show the 2009/2020 dips?


In [5]:
def run_adf(series, label):
    result = adfuller(series.dropna())
    print(f"\nADF test - {label}")
    print(f"  ADF statistic: {result[0]:.4f}")
    print(f"  p-value:       {result[1]:.4f}")
    print(f"  -> {'STATIONARY (reject H0)' if result[1] < 0.05 else 'NOT stationary (fail to reject H0)'} at 5%")


# Stationarity: KPSS (H0 = series IS stationary - the opposite null of ADF).
# Run both: if they agree, easy call. If they disagree (commonly ADF says
# stationary but KPSS says not), that usually flags trend-stationarity -
# stationary around a deterministic trend rather than a pure random walk -
# which means detrending is the right fix, not differencing.
def run_kpss(series, label, regression="c"):
    stat, p_value, lags, crit = kpss(series.dropna(), regression=regression, nlags="auto")
    print(f"\nKPSS test - {label} (regression='{regression}')")
    print(f"  KPSS statistic: {stat:.4f}")
    print(f"  p-value:        {p_value:.4f}  (statsmodels clips this to [0.01, 0.10])")
    print(f"  -> {'NOT stationary (reject H0)' if p_value < 0.05 else 'STATIONARY (fail to reject H0)'} at 5%")


print("=" * 60)
print("ADF tests")
print("=" * 60)
run_adf(df["Headcount"], "Headcount (raw)")
run_adf(df["Headcount"].diff(), "Headcount (1st difference)")
run_adf(df["Attrition_Rate_Pct"], "Attrition_Rate_Pct (raw)")

print("\n" + "=" * 60)
print("KPSS tests (cross-check against ADF above)")
print("=" * 60)
run_kpss(df["Headcount"], "Headcount (raw)")
run_kpss(df["Headcount"].diff(), "Headcount (1st difference)")
run_kpss(df["Attrition_Rate_Pct"], "Attrition_Rate_Pct (raw)")

ADF tests

ADF test - Headcount (raw)
  ADF statistic: 1.5329
  p-value:       0.9976
  -> NOT stationary (fail to reject H0) at 5%

ADF test - Headcount (1st difference)
  ADF statistic: -5.1378
  p-value:       0.0000
  -> STATIONARY (reject H0) at 5%

ADF test - Attrition_Rate_Pct (raw)
  ADF statistic: -3.8299
  p-value:       0.0026
  -> STATIONARY (reject H0) at 5%

KPSS tests (cross-check against ADF above)

KPSS test - Headcount (raw) (regression='c')
  KPSS statistic: 2.2347
  p-value:        0.0100  (statsmodels clips this to [0.01, 0.10])
  -> NOT stationary (reject H0) at 5%

KPSS test - Headcount (1st difference) (regression='c')
  KPSS statistic: 0.2793
  p-value:        0.1000  (statsmodels clips this to [0.01, 0.10])
  -> STATIONARY (fail to reject H0) at 5%

KPSS test - Attrition_Rate_Pct (raw) (regression='c')
  KPSS statistic: 1.1351
  p-value:        0.0100  (statsmodels clips this to [0.01, 0.10])
  -> NOT stationary (reject H0) at 5%


/sessions/wizardly-confident-brahmagupta/tmp/ipykernel_15/2311317419.py:15: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  stat, p_value, lags, crit = kpss(series.dropna(), regression=regression, nlags="auto")
/sessions/wizardly-confident-brahmagupta/tmp/ipykernel_15/2311317419.py:15: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  stat, p_value, lags, crit = kpss(series.dropna(), regression=regression, nlags="auto")
/sessions/wizardly-confident-brahmagupta/tmp/ipykernel_15/2311317419.py:15: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  stat, p_value, lags, crit = kpss(series.dropna(), regression=regression, nlags="auto")


In [6]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
plot_acf(df["Attrition_Rate_Pct"].dropna(), ax=axes[0, 0], lags=36, title="ACF - Attrition Rate (raw)")
plot_pacf(df["Attrition_Rate_Pct"].dropna(), ax=axes[0, 1], lags=36, title="PACF - Attrition Rate (raw)")
plot_acf(df["Headcount"].diff().dropna(), ax=axes[1, 0], lags=36, title="ACF - Headcount (1st diff)")
plot_pacf(df["Headcount"].diff().dropna(), ax=axes[1, 1], lags=36, title="PACF - Headcount (1st diff)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "03_acf_pacf.png"), dpi=150)
plt.close()
print("\nSaved 03_acf_pacf.png - look for: how many lags stick out above the confidence band before "
      "cutting off, and whether there's a spike around lag 12 (seasonal signal).")

print("\nDone. Figures are in results/figures/. Report back: the two ADF p-values above, "
      "and what you see in 01_raw_series.png / 03_acf_pacf.png (trend? seasonality? spikes at which lags?).")


Saved 03_acf_pacf.png - look for: how many lags stick out above the confidence band before cutting off, and whether there's a spike around lag 12 (seasonal signal).

Done. Figures are in results/figures/. Report back: the two ADF p-values above, and what you see in 01_raw_series.png / 03_acf_pacf.png (trend? seasonality? spikes at which lags?).
